In [1]:
import nltk 
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import random

In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/gsri-18/ISEAR-dataset-complete/ISEAR_dataset_complete.csv")
df = df.drop(columns=["Unnamed: 2"])
content = df["content"]
print(content.head())
emotions = df["emotion"].unique()

0    On days when I feel close to my partner and ot...
1    Every time I imagine that someone I love or I ...
2    When I had been obviously unjustly treated and...
3    When I think about the short time that we live...
4    At a gathering I found myself involuntarily si...
Name: content, dtype: str


In [3]:
# Dataset statistics before tokenization
emotions_unique = df["emotion"].unique()
print("Dataset statistics before tokenization for each unique emotion:")
print()
for emotion in emotions_unique:
    subset = df[df["emotion"] == emotion]
    N = len(subset["emotion"])
    print(f"{emotion} has {N} occurences, {round(N/len(df["emotion"]) * 100, 2)}%")
    average_length = subset["content"].str.len().mean()
    print(f"{emotion} has an average length of {average_length:.2f} characters")
    print()


Dataset statistics before tokenization for each unique emotion:

joy has 1092 occurences, 14.53%
joy has an average length of 103.14 characters

fear has 1076 occurences, 14.32%
fear has an average length of 123.96 characters

anger has 1079 occurences, 14.36%
anger has an average length of 130.65 characters

sadness has 1082 occurences, 14.4%
sadness has an average length of 103.93 characters

disgust has 1066 occurences, 14.18%
disgust has an average length of 114.61 characters

shame has 1071 occurences, 14.25%
shame has an average length of 116.73 characters

guilt has 1050 occurences, 13.97%
guilt has an average length of 124.89 characters



In [4]:
# Representation 1: Tokenization
tokens = []
for element in df["content"]:
    tokenized_sentence = word_tokenize(element)
    tokens.append(tokenized_sentence)
df["tokens"] = tokens


In [5]:
df.head(5)
# print avarage token length of sentence here.

,emotion,content,tokens
0,joy,On days when I feel close to my partner and ot...,"[On, days, when, I, feel, close, to, my, partn..."
1,fear,Every time I imagine that someone I love or I ...,"[Every, time, I, imagine, that, someone, I, lo..."
2,anger,When I had been obviously unjustly treated and...,"[When, I, had, been, obviously, unjustly, trea..."
3,sadness,When I think about the short time that we live...,"[When, I, think, about, the, short, time, that..."
4,disgust,At a gathering I found myself involuntarily si...,"[At, a, gathering, I, found, myself, involunta..."


In [6]:
# Representation 2: Bag of words
vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)

bag_matrix = vectorizer.fit_transform(df["tokens"].tolist())

bag_df = pd.DataFrame(bag_matrix.toarray(),
                      columns=vectorizer.get_feature_names_out(),
                      index=df.index
)

print(bag_df.head(5))

   !  $  %  &  '  ''  'd  'll  'm  're  ...  yr.  yrs  yukky  zero  \
0  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
1  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
2  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
3  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   
4  0  0  0  0  0   0   0    0   0    0  ...    0    0      0     0   

   zig-zagging  zip  zipper  zombies  zone  zoophiliac  
0            0    0       0        0     0           0  
1            0    0       0        0     0           0  
2            0    0       0        0     0           0  
3            0    0       0        0     0           0  
4            0    0       0        0     0           0  

[5 rows x 10173 columns]


In [7]:
# Representation 3: Bigrams

bigrams = []
for sentence in df["tokens"]:
    bigrams_per_sentence = []
    for i in range(len(sentence) - 1):
        bigram = sentence[i] + " " + sentence[i+1]
        bigrams_per_sentence.append(bigram)
    bigrams.append(bigrams_per_sentence)
df["bigrams"] = bigrams

In [8]:
# Representation 4: Trigrams

trigrams = []
for sentence in df["tokens"]:
    trigrams_per_sentence = []
    for i in range(len(sentence) - 2):
        trigram = sentence[i] + " " + sentence[i+1] + " " + sentence[i+2]
        trigrams_per_sentence.append(trigram)
    trigrams.append(trigrams_per_sentence)
df["trigrams"] = trigrams

In [9]:
print(df["trigrams"][0])

['On days when', 'days when I', 'when I feel', 'I feel close', 'feel close to', 'close to my', 'to my partner', 'my partner and', 'partner and other', 'and other friends', 'other friends .', 'friends . When', '. When I', 'When I feel', 'I feel at', 'feel at peace', 'at peace with', 'peace with myself', 'with myself and', 'myself and also', 'and also experience', 'also experience a', 'experience a close', 'a close contact', 'close contact with', 'contact with people', 'with people whom', 'people whom I', 'whom I regard', 'I regard greatly', 'regard greatly .']


In [10]:
# Architecture 1: Log-Linear
import itertools
nltk.download("punkt_tab")

corpus = " ".join(df["content"].fillna("").astype(str))

vocab = sorted(set(word_tokenize(corpus)))
token_to_id = {token: i for i, token in enumerate(vocab)}

class Encoder:
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")

def cross_validation_split(pairs, k, seed=0):
    """
    Shuffles a list of labelled texts and splits the result into k folds.
    
    pairs: lists of tuples (text, label)
    k: number of folds
    seed: for reproducible shuffling
    """
    
    # Shuffle the pairs
    random.seed(seed)
    pairs = list(pairs)
    ids = list(range(len(pairs)))
    random.shuffle(ids)

    # Split without coercing (text, int) tuples into a string array
    ids_parts = np.array_split(np.array(ids, dtype=object), k)
    
    # Create a dictionary to store the training and validation sets for each fold
    cross_validation_sets = {}
    for i in range(k):
        validation_set = [pairs[j] for j in ids_parts[i]] 
        training_ids = list(itertools.chain.from_iterable(part for k, part in enumerate(ids_parts) if k != i))
        training_set = [pairs[j] for j in training_ids]
        cross_validation_sets[i] = {
            "training": training_set, 
            "validation": validation_set, 
            'training_ids': training_ids,   # these are not often needed, but we return it just in case
            'validation_ids': ids_parts[i]  # these are not often needed, but we return it just in case
        }

    return cross_validation_sets

my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

# for seed in my_seeds:  
#     my_cross_validation_sets.append(
#         cross_validation_split(
#             # we pair the texts with their labels
#             pairs=list(zip(dataset["train"]["text"], dataset["train"]["label"])), 
#             k=num_folds,
#             seed=seed
#         )
#     )

class TokenIndicator(Encoder):
    def __init__(self, token_to_id):
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.token_to_id)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self), dtype=np.float32)

        for token in set(word_tokenize(x)):
            token_id = self.token_to_id.get(token)
            if token_id is not None:
                vector[token_id] = 1

        return vector
    
token_indicator_encoder = TokenIndicator(token_to_id)
doc_repr = token_indicator_encoder(corpus)

print(len(doc_repr))
print(doc_repr.sum())

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/ianhutcheson/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


10176
10176.0


In [11]:
import os
results_folder = "../Data/Results"
os.makedirs(results_folder, exist_ok=True)

In [ ]:
# Architecture 2: Single hidden layer MLP
import smlp

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)
accuracies = []
for fold in cv:
    metrics, best_epoch, best_pred = smlp.train_neural_network_model(
    cv[fold]["training"],
    cv[fold]["validation"],
    num_classes=len(labels),
    encoder=token_indicator_encoder,
    save_to=f"{results_folder}/mlp_{fold}_test",
    num_epochs=10)
    accuracies.append(best_pred)
print(accuracies)

Transforming documents...
Document transformation done! Time taken: 1.325 seconds
Dataloader done! Time taken: 0.281 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7961 | train acc: 0.3984 | val loss: 1.5578 | val acc: 0.5226
Epoch 02 | train loss: 1.1656 | train acc: 0.6801 | val loss: 1.2447 | val acc: 0.5665
Epoch 03 | train loss: 0.7310 | train acc: 0.8012 | val loss: 1.1815 | val acc: 0.5818
Epoch 04 | train loss: 0.4991 | train acc: 0.8733 | val loss: 1.1993 | val acc: 0.5798
Epoch 05 | train loss: 0.3581 | train acc: 0.9143 | val loss: 1.2501 | val acc: 0.5672
Epoch 06 | train loss: 0.2657 | train acc: 0.9385 | val loss: 1.3149 | val acc: 0.5824
Epoch 07 | train loss: 0.2021 | train acc: 0.9587 | val loss: 1.3780 | val acc: 0.5745
Epoch 08 | train loss: 0.1561 | train acc: 0.9704 | val loss: 1.4500 | val acc: 0.5778
Epoch 09 | train loss: 0.1254 | train acc: 0.9769 | val loss: 1.5303 | val acc: 0.5665
Epoch 10 | train loss: 0.1010 | train acc: 0.9824 | val loss: 1.6004 | val acc: 0.5652
Training done! Time taken: 9.727 seconds
Transforming documents...
Document transformation done! Time taken: 1.305 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7933 | train acc: 0.3931 | val loss: 1.5362 | val acc: 0.5436
Epoch 02 | train loss: 1.1678 | train acc: 0.6777 | val loss: 1.2296 | val acc: 0.5822
Epoch 03 | train loss: 0.7418 | train acc: 0.8001 | val loss: 1.1672 | val acc: 0.5988
Epoch 04 | train loss: 0.5072 | train acc: 0.8691 | val loss: 1.1867 | val acc: 0.5848
Epoch 05 | train loss: 0.3668 | train acc: 0.9129 | val loss: 1.2226 | val acc: 0.5802
Epoch 06 | train loss: 0.2714 | train acc: 0.9401 | val loss: 1.2802 | val acc: 0.5709
Epoch 07 | train loss: 0.2083 | train acc: 0.9559 | val loss: 1.3531 | val acc: 0.5675
Epoch 08 | train loss: 0.1627 | train acc: 0.9652 | val loss: 1.4014 | val acc: 0.5742
Epoch 09 | train loss: 0.1308 | train acc: 0.9775 | val loss: 1.4692 | val acc: 0.5655
Epoch 10 | train loss: 0.1063 | train acc: 0.9825 | val loss: 1.5400 | val acc: 0.5669
Training done! Time taken: 9.539 seconds
Transforming documents...
Document transformation done! Time taken: 1.345 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7950 | train acc: 0.3922 | val loss: 1.5605 | val acc: 0.5156
Epoch 02 | train loss: 1.1662 | train acc: 0.6735 | val loss: 1.2219 | val acc: 0.5875
Epoch 03 | train loss: 0.7377 | train acc: 0.8004 | val loss: 1.1657 | val acc: 0.5975
Epoch 04 | train loss: 0.5062 | train acc: 0.8729 | val loss: 1.1944 | val acc: 0.5868
Epoch 05 | train loss: 0.3596 | train acc: 0.9167 | val loss: 1.2338 | val acc: 0.5955
Epoch 06 | train loss: 0.2653 | train acc: 0.9438 | val loss: 1.3059 | val acc: 0.5835
Epoch 07 | train loss: 0.2033 | train acc: 0.9568 | val loss: 1.3773 | val acc: 0.5822
Epoch 08 | train loss: 0.1548 | train acc: 0.9724 | val loss: 1.4505 | val acc: 0.5755
Epoch 09 | train loss: 0.1211 | train acc: 0.9805 | val loss: 1.5291 | val acc: 0.5762
Epoch 10 | train loss: 0.0989 | train acc: 0.9850 | val loss: 1.6058 | val acc: 0.5695
Training done! Time taken: 9.756 seconds
Transforming documents...
Document transformation done! Time taken: 1.300 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7959 | train acc: 0.3858 | val loss: 1.5637 | val acc: 0.5323
Epoch 02 | train loss: 1.1717 | train acc: 0.6785 | val loss: 1.2380 | val acc: 0.5828
Epoch 03 | train loss: 0.7398 | train acc: 0.7978 | val loss: 1.1643 | val acc: 0.5882
Epoch 04 | train loss: 0.5023 | train acc: 0.8714 | val loss: 1.1829 | val acc: 0.5928
Epoch 05 | train loss: 0.3617 | train acc: 0.9102 | val loss: 1.2324 | val acc: 0.5755
Epoch 06 | train loss: 0.2676 | train acc: 0.9410 | val loss: 1.2981 | val acc: 0.5755
Epoch 07 | train loss: 0.2036 | train acc: 0.9591 | val loss: 1.3649 | val acc: 0.5802
Epoch 08 | train loss: 0.1584 | train acc: 0.9687 | val loss: 1.4358 | val acc: 0.5655
Epoch 09 | train loss: 0.1263 | train acc: 0.9762 | val loss: 1.5082 | val acc: 0.5635
Epoch 10 | train loss: 0.1023 | train acc: 0.9819 | val loss: 1.5839 | val acc: 0.5615
Training done! Time taken: 9.608 seconds
Transforming documents...
Document transformation done! Time taken: 1.309 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8059 | train acc: 0.3795 | val loss: 1.5595 | val acc: 0.5396
Epoch 02 | train loss: 1.1876 | train acc: 0.6666 | val loss: 1.2049 | val acc: 0.5862
Epoch 03 | train loss: 0.7457 | train acc: 0.7958 | val loss: 1.1323 | val acc: 0.5968
Epoch 04 | train loss: 0.5152 | train acc: 0.8690 | val loss: 1.1356 | val acc: 0.5995
Epoch 05 | train loss: 0.3700 | train acc: 0.9115 | val loss: 1.1841 | val acc: 0.5828
Epoch 06 | train loss: 0.2747 | train acc: 0.9391 | val loss: 1.2413 | val acc: 0.5749
Epoch 07 | train loss: 0.2094 | train acc: 0.9581 | val loss: 1.2974 | val acc: 0.5775
Epoch 08 | train loss: 0.1647 | train acc: 0.9689 | val loss: 1.3717 | val acc: 0.5762
Epoch 09 | train loss: 0.1307 | train acc: 0.9767 | val loss: 1.4396 | val acc: 0.5709
Epoch 10 | train loss: 0.1057 | train acc: 0.9827 | val loss: 1.5015 | val acc: 0.5622
Training done! Time taken: 10.132 seconds
[array([1, 2, 1, ..., 3, 5, 0], shape=(1504,)), array([6, 2, 2, ..., 5, 0, 4], shape=(15

In [ ]:
# Architecture 3: Modified MLP - 2 hidden layers
import smlp_mod
import itertools

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)
accuracies = []
for fold in cv:
    metrics, best_epoch, best_pred = smlp_mod.train_neural_network_model(
    cv[fold]["training"],
    cv[fold]["validation"],
    num_classes=len(labels),
    hidden_dim=64,
    hidden_dim2=128,
    encoder=token_indicator_encoder,
    save_to=f"{results_folder}/mlp_{fold}_test",
    num_epochs=10)
    accuracies.append(best_pred)
print(accuracies)

Transforming documents...
Document transformation done! Time taken: 1.353 seconds
Dataloader done! Time taken: 0.338 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8451 | train acc: 0.3107 | val loss: 1.5652 | val acc: 0.4967
Epoch 02 | train loss: 1.1300 | train acc: 0.6437 | val loss: 1.2066 | val acc: 0.5625
Epoch 03 | train loss: 0.6186 | train acc: 0.8092 | val loss: 1.2289 | val acc: 0.5818
Epoch 04 | train loss: 0.3667 | train acc: 0.8942 | val loss: 1.3647 | val acc: 0.5771
Epoch 05 | train loss: 0.2314 | train acc: 0.9353 | val loss: 1.5068 | val acc: 0.5731
Epoch 06 | train loss: 0.1469 | train acc: 0.9614 | val loss: 1.6724 | val acc: 0.5632
Epoch 07 | train loss: 0.1051 | train acc: 0.9757 | val loss: 1.8303 | val acc: 0.5618
Epoch 08 | train loss: 0.0755 | train acc: 0.9812 | val loss: 1.9735 | val acc: 0.5585
Epoch 09 | train loss: 0.0519 | train acc: 0.9890 | val loss: 2.0939 | val acc: 0.5612
Epoch 10 | train loss: 0.0429 | train acc: 0.9900 | val loss: 2.2475 | val acc: 0.5525
Training done! Time taken: 6.358 seconds
Transforming documents...
Document transformation done! Time taken: 1.303 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8429 | train acc: 0.2887 | val loss: 1.5602 | val acc: 0.4731
Epoch 02 | train loss: 1.1631 | train acc: 0.6316 | val loss: 1.1976 | val acc: 0.5875
Epoch 03 | train loss: 0.6538 | train acc: 0.7944 | val loss: 1.2296 | val acc: 0.5862
Epoch 04 | train loss: 0.3885 | train acc: 0.8904 | val loss: 1.3209 | val acc: 0.5768
Epoch 05 | train loss: 0.2581 | train acc: 0.9268 | val loss: 1.4617 | val acc: 0.5655
Epoch 06 | train loss: 0.1712 | train acc: 0.9593 | val loss: 1.5673 | val acc: 0.5635
Epoch 07 | train loss: 0.1164 | train acc: 0.9711 | val loss: 1.7434 | val acc: 0.5695
Epoch 08 | train loss: 0.0894 | train acc: 0.9807 | val loss: 1.8403 | val acc: 0.5602
Epoch 09 | train loss: 0.0678 | train acc: 0.9845 | val loss: 1.9651 | val acc: 0.5629
Epoch 10 | train loss: 0.0562 | train acc: 0.9872 | val loss: 2.0770 | val acc: 0.5642
Training done! Time taken: 6.286 seconds
Transforming documents...
Document transformation done! Time taken: 1.301 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8428 | train acc: 0.3168 | val loss: 1.5574 | val acc: 0.4910
Epoch 02 | train loss: 1.1459 | train acc: 0.6308 | val loss: 1.1974 | val acc: 0.5882
Epoch 03 | train loss: 0.6569 | train acc: 0.7901 | val loss: 1.2363 | val acc: 0.5842
Epoch 04 | train loss: 0.3986 | train acc: 0.8879 | val loss: 1.3439 | val acc: 0.5815
Epoch 05 | train loss: 0.2513 | train acc: 0.9320 | val loss: 1.4991 | val acc: 0.5729
Epoch 06 | train loss: 0.1621 | train acc: 0.9594 | val loss: 1.6552 | val acc: 0.5722
Epoch 07 | train loss: 0.1066 | train acc: 0.9756 | val loss: 1.8089 | val acc: 0.5689
Epoch 08 | train loss: 0.0744 | train acc: 0.9837 | val loss: 1.9665 | val acc: 0.5675
Epoch 09 | train loss: 0.0526 | train acc: 0.9890 | val loss: 2.1131 | val acc: 0.5609
Epoch 10 | train loss: 0.0398 | train acc: 0.9930 | val loss: 2.2303 | val acc: 0.5556
Training done! Time taken: 6.309 seconds
Transforming documents...
Document transformation done! Time taken: 1.310 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8387 | train acc: 0.3047 | val loss: 1.5502 | val acc: 0.4797
Epoch 02 | train loss: 1.1530 | train acc: 0.6222 | val loss: 1.2171 | val acc: 0.5622
Epoch 03 | train loss: 0.6473 | train acc: 0.8013 | val loss: 1.2067 | val acc: 0.5795
Epoch 04 | train loss: 0.3925 | train acc: 0.8889 | val loss: 1.3327 | val acc: 0.5808
Epoch 05 | train loss: 0.2466 | train acc: 0.9323 | val loss: 1.4523 | val acc: 0.5815
Epoch 06 | train loss: 0.1646 | train acc: 0.9559 | val loss: 1.6164 | val acc: 0.5709
Epoch 07 | train loss: 0.1104 | train acc: 0.9731 | val loss: 1.7647 | val acc: 0.5622
Epoch 08 | train loss: 0.0783 | train acc: 0.9819 | val loss: 1.9343 | val acc: 0.5662
Epoch 09 | train loss: 0.0569 | train acc: 0.9884 | val loss: 2.0811 | val acc: 0.5635
Epoch 10 | train loss: 0.0470 | train acc: 0.9887 | val loss: 2.1747 | val acc: 0.5496
Training done! Time taken: 6.368 seconds
Transforming documents...
Document transformation done! Time taken: 1.281 seconds
Dataloa

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.8361 | train acc: 0.3359 | val loss: 1.5147 | val acc: 0.4950
Epoch 02 | train loss: 1.1296 | train acc: 0.6285 | val loss: 1.1637 | val acc: 0.5908
Epoch 03 | train loss: 0.6482 | train acc: 0.7988 | val loss: 1.1817 | val acc: 0.5848
Epoch 04 | train loss: 0.4058 | train acc: 0.8786 | val loss: 1.2808 | val acc: 0.5828
Epoch 05 | train loss: 0.2631 | train acc: 0.9247 | val loss: 1.4342 | val acc: 0.5682
Epoch 06 | train loss: 0.1698 | train acc: 0.9558 | val loss: 1.5670 | val acc: 0.5615
Epoch 07 | train loss: 0.1135 | train acc: 0.9732 | val loss: 1.7182 | val acc: 0.5569
Epoch 08 | train loss: 0.0826 | train acc: 0.9792 | val loss: 1.8435 | val acc: 0.5562
Epoch 09 | train loss: 0.0599 | train acc: 0.9864 | val loss: 2.0117 | val acc: 0.5522
Epoch 10 | train loss: 0.0517 | train acc: 0.9890 | val loss: 2.1032 | val acc: 0.5476
Training done! Time taken: 6.248 seconds
[array([4, 2, 3, ..., 3, 5, 5], shape=(1504,)), array([6, 2, 2, ..., 5, 0, 4], shape=(150

In [17]:
from collections import Counter
import numpy as np
from nltk.tokenize import word_tokenize


def ngram_features(text):
    tokens = [token.lower() for token in word_tokenize(text)]

    unigrams = {("uni", token) for token in tokens}
    bigrams = {
        ("bi", tokens[i], tokens[i + 1])
        for i in range(len(tokens) - 1)
    }

    return unigrams | bigrams


class NgramIndicator(Encoder):
    def __init__(self, feature_to_id):
        self.feature_to_id = feature_to_id

    def __len__(self):
        return len(self.feature_to_id)

    def __call__(self, text):
        vector = np.zeros(len(self), dtype=np.float32)

        for feature in ngram_features(text):
            feature_id = self.feature_to_id.get(feature)
            if feature_id is not None:
                vector[feature_id] = 1

        return vector


def make_ngram_encoder(training_texts, min_df=2, max_features=50_000):
    # Count how many training documents contain each feature.
    document_frequency = Counter()
    for text in training_texts:
        document_frequency.update(ngram_features(text))

    features = [
        (feature, count)
        for feature, count in document_frequency.items()
        if count >= min_df
    ]
    features.sort(key=lambda item: (-item[1], repr(item[0])))

    if max_features is not None:
        features = features[:max_features]

    feature_to_id = {
        feature: i for i, (feature, _) in enumerate(features)
    }
    return NgramIndicator(feature_to_id)

In [19]:
# Architecture 2: Single hidden layer MLP (Bigrams)
import smlp

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)
accuracies = []
for fold in cv:
    training = cv[fold]["training"]
    validation = cv[fold]["validation"]

    encoder = make_ngram_encoder(
        (text for text, label in training)
    )

    metrics, best_epoch, best_pred = smlp_mod.train_neural_network_model(
        training,
        validation,
        num_classes=len(labels),
        encoder=encoder,
        save_to=f"{results_folder}/mlp_{fold}",
        num_epochs=10,
    )
    accuracies.append(best_pred)
print(accuracies)

Transforming documents...
Document transformation done! Time taken: 1.498 seconds
Dataloader done! Time taken: 0.538 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7219 | train acc: 0.3994 | val loss: 1.3458 | val acc: 0.5379
Epoch 02 | train loss: 0.7760 | train acc: 0.7596 | val loss: 1.2034 | val acc: 0.5851
Epoch 03 | train loss: 0.2766 | train acc: 0.9281 | val loss: 1.3724 | val acc: 0.5824
Epoch 04 | train loss: 0.1076 | train acc: 0.9777 | val loss: 1.5723 | val acc: 0.5751
Epoch 05 | train loss: 0.0512 | train acc: 0.9915 | val loss: 1.7398 | val acc: 0.5612
Epoch 06 | train loss: 0.0305 | train acc: 0.9955 | val loss: 1.8648 | val acc: 0.5625
Epoch 07 | train loss: 0.0202 | train acc: 0.9965 | val loss: 1.9787 | val acc: 0.5638
Epoch 08 | train loss: 0.0168 | train acc: 0.9967 | val loss: 2.0602 | val acc: 0.5632
Epoch 09 | train loss: 0.0136 | train acc: 0.9970 | val loss: 2.1581 | val acc: 0.5632
Epoch 10 | train loss: 0.0127 | train acc: 0.9965 | val loss: 2.2026 | val acc: 0.5625
Training done! Time taken: 15.766 seconds
Transforming documents...
Document transformation done! Time taken: 1.387 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7373 | train acc: 0.3812 | val loss: 1.3459 | val acc: 0.5223
Epoch 02 | train loss: 0.8305 | train acc: 0.7304 | val loss: 1.1880 | val acc: 0.5908
Epoch 03 | train loss: 0.3008 | train acc: 0.9223 | val loss: 1.3251 | val acc: 0.5935
Epoch 04 | train loss: 0.1177 | train acc: 0.9746 | val loss: 1.4808 | val acc: 0.5935
Epoch 05 | train loss: 0.0581 | train acc: 0.9923 | val loss: 1.6612 | val acc: 0.5768
Epoch 06 | train loss: 0.0344 | train acc: 0.9947 | val loss: 1.7625 | val acc: 0.5808
Epoch 07 | train loss: 0.0234 | train acc: 0.9960 | val loss: 1.8916 | val acc: 0.5775
Epoch 08 | train loss: 0.0192 | train acc: 0.9958 | val loss: 1.9589 | val acc: 0.5755
Epoch 09 | train loss: 0.0141 | train acc: 0.9977 | val loss: 2.0235 | val acc: 0.5755
Epoch 10 | train loss: 0.0110 | train acc: 0.9983 | val loss: 2.1177 | val acc: 0.5689
Training done! Time taken: 15.098 seconds
Transforming documents...
Document transformation done! Time taken: 1.410 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7306 | train acc: 0.3699 | val loss: 1.3257 | val acc: 0.5496
Epoch 02 | train loss: 0.7941 | train acc: 0.7567 | val loss: 1.1915 | val acc: 0.5862
Epoch 03 | train loss: 0.2775 | train acc: 0.9255 | val loss: 1.3634 | val acc: 0.5842
Epoch 04 | train loss: 0.1089 | train acc: 0.9785 | val loss: 1.5376 | val acc: 0.5788
Epoch 05 | train loss: 0.0518 | train acc: 0.9915 | val loss: 1.7199 | val acc: 0.5762
Epoch 06 | train loss: 0.0300 | train acc: 0.9955 | val loss: 1.8404 | val acc: 0.5795
Epoch 07 | train loss: 0.0187 | train acc: 0.9973 | val loss: 1.9919 | val acc: 0.5709
Epoch 08 | train loss: 0.0144 | train acc: 0.9975 | val loss: 2.0727 | val acc: 0.5755
Epoch 09 | train loss: 0.0118 | train acc: 0.9975 | val loss: 2.1487 | val acc: 0.5709
Epoch 10 | train loss: 0.0110 | train acc: 0.9978 | val loss: 2.2006 | val acc: 0.5742
Training done! Time taken: 15.611 seconds
Transforming documents...
Document transformation done! Time taken: 1.472 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7308 | train acc: 0.3586 | val loss: 1.3699 | val acc: 0.5343
Epoch 02 | train loss: 0.8064 | train acc: 0.7512 | val loss: 1.1804 | val acc: 0.5921
Epoch 03 | train loss: 0.2847 | train acc: 0.9272 | val loss: 1.2895 | val acc: 0.5955
Epoch 04 | train loss: 0.1107 | train acc: 0.9769 | val loss: 1.4527 | val acc: 0.5695
Epoch 05 | train loss: 0.0539 | train acc: 0.9910 | val loss: 1.6026 | val acc: 0.5675
Epoch 06 | train loss: 0.0324 | train acc: 0.9950 | val loss: 1.7281 | val acc: 0.5702
Epoch 07 | train loss: 0.0211 | train acc: 0.9967 | val loss: 1.8458 | val acc: 0.5662
Epoch 08 | train loss: 0.0167 | train acc: 0.9967 | val loss: 1.9343 | val acc: 0.5609
Epoch 09 | train loss: 0.0129 | train acc: 0.9975 | val loss: 2.0159 | val acc: 0.5655
Epoch 10 | train loss: 0.0100 | train acc: 0.9980 | val loss: 2.0905 | val acc: 0.5622
Training done! Time taken: 15.330 seconds
Transforming documents...
Document transformation done! Time taken: 1.414 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7438 | train acc: 0.3883 | val loss: 1.3367 | val acc: 0.5369
Epoch 02 | train loss: 0.8143 | train acc: 0.7449 | val loss: 1.1222 | val acc: 0.6041
Epoch 03 | train loss: 0.2768 | train acc: 0.9262 | val loss: 1.2623 | val acc: 0.5915
Epoch 04 | train loss: 0.1080 | train acc: 0.9759 | val loss: 1.3876 | val acc: 0.5961
Epoch 05 | train loss: 0.0525 | train acc: 0.9907 | val loss: 1.5198 | val acc: 0.5948
Epoch 06 | train loss: 0.0309 | train acc: 0.9952 | val loss: 1.6249 | val acc: 0.5988
Epoch 07 | train loss: 0.0215 | train acc: 0.9965 | val loss: 1.7028 | val acc: 0.5955
Epoch 08 | train loss: 0.0163 | train acc: 0.9975 | val loss: 1.7813 | val acc: 0.5915
Epoch 09 | train loss: 0.0136 | train acc: 0.9970 | val loss: 1.8621 | val acc: 0.5808
Epoch 10 | train loss: 0.0114 | train acc: 0.9975 | val loss: 1.9168 | val acc: 0.5815
Training done! Time taken: 14.429 seconds
[array([0, 2, 6, ..., 3, 5, 0], shape=(1504,)), array([6, 2, 2, ..., 5, 0, 4], shape=(15

In [ ]:
# Architecture 3: Modified MLP - 2 hidden layers (Bigrams)
import smlp_mod
import itertools

labels = sorted(df["emotion"].unique())
pairs = [(content, labels.index(emotion)) for content, emotion in zip(df["content"], df["emotion"])]
cv = cross_validation_split(pairs, k=num_folds, seed=0)
accuracies = []
for fold in cv:
    training = cv[fold]["training"]
    validation = cv[fold]["validation"]

    encoder = make_ngram_encoder(
        (text for text, label in training)
    )

    metrics, best_epoch, best_pred = smlp_mod.train_neural_network_model(
        training,
        validation,
        num_classes=len(labels),
        encoder=encoder,
        save_to=f"{results_folder}/mlp_{fold}",
        num_epochs=10,
    )
    accuracies.append(best_pred)
print(accuracies)

Transforming documents...
Document transformation done! Time taken: 1.482 seconds
Dataloader done! Time taken: 0.569 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7366 | train acc: 0.3723 | val loss: 1.3487 | val acc: 0.5273
Epoch 02 | train loss: 0.7948 | train acc: 0.7522 | val loss: 1.1965 | val acc: 0.5871
Epoch 03 | train loss: 0.2718 | train acc: 0.9318 | val loss: 1.3613 | val acc: 0.5785
Epoch 04 | train loss: 0.1076 | train acc: 0.9749 | val loss: 1.5520 | val acc: 0.5711
Epoch 05 | train loss: 0.0507 | train acc: 0.9912 | val loss: 1.6949 | val acc: 0.5698
Epoch 06 | train loss: 0.0298 | train acc: 0.9960 | val loss: 1.8418 | val acc: 0.5705
Epoch 07 | train loss: 0.0222 | train acc: 0.9963 | val loss: 1.9328 | val acc: 0.5638
Epoch 08 | train loss: 0.0176 | train acc: 0.9962 | val loss: 2.0616 | val acc: 0.5545
Epoch 09 | train loss: 0.0155 | train acc: 0.9973 | val loss: 2.1209 | val acc: 0.5598
Epoch 10 | train loss: 0.0121 | train acc: 0.9970 | val loss: 2.1899 | val acc: 0.5578
Training done! Time taken: 16.115 seconds
Transforming documents...
Document transformation done! Time taken: 1.475 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7460 | train acc: 0.3655 | val loss: 1.3860 | val acc: 0.5196
Epoch 02 | train loss: 0.8403 | train acc: 0.7416 | val loss: 1.1718 | val acc: 0.5968
Epoch 03 | train loss: 0.3012 | train acc: 0.9197 | val loss: 1.3020 | val acc: 0.5941
Epoch 04 | train loss: 0.1225 | train acc: 0.9736 | val loss: 1.4644 | val acc: 0.5935
Epoch 05 | train loss: 0.0585 | train acc: 0.9889 | val loss: 1.6162 | val acc: 0.5908
Epoch 06 | train loss: 0.0360 | train acc: 0.9932 | val loss: 1.7474 | val acc: 0.5762
Epoch 07 | train loss: 0.0254 | train acc: 0.9955 | val loss: 1.8597 | val acc: 0.5822
Epoch 08 | train loss: 0.0185 | train acc: 0.9963 | val loss: 1.9623 | val acc: 0.5768
Epoch 09 | train loss: 0.0154 | train acc: 0.9970 | val loss: 2.0106 | val acc: 0.5808
Epoch 10 | train loss: 0.0132 | train acc: 0.9970 | val loss: 2.0539 | val acc: 0.5795
Training done! Time taken: 15.197 seconds
Transforming documents...
Document transformation done! Time taken: 1.463 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7288 | train acc: 0.3732 | val loss: 1.3592 | val acc: 0.5449
Epoch 02 | train loss: 0.7960 | train acc: 0.7560 | val loss: 1.2062 | val acc: 0.5795
Epoch 03 | train loss: 0.2779 | train acc: 0.9275 | val loss: 1.3477 | val acc: 0.5822
Epoch 04 | train loss: 0.1058 | train acc: 0.9812 | val loss: 1.5690 | val acc: 0.5702
Epoch 05 | train loss: 0.0511 | train acc: 0.9917 | val loss: 1.7304 | val acc: 0.5762
Epoch 06 | train loss: 0.0289 | train acc: 0.9963 | val loss: 1.8572 | val acc: 0.5709
Epoch 07 | train loss: 0.0194 | train acc: 0.9973 | val loss: 1.9491 | val acc: 0.5729
Epoch 08 | train loss: 0.0144 | train acc: 0.9975 | val loss: 2.0460 | val acc: 0.5649
Epoch 09 | train loss: 0.0118 | train acc: 0.9978 | val loss: 2.1329 | val acc: 0.5675
Epoch 10 | train loss: 0.0101 | train acc: 0.9982 | val loss: 2.2105 | val acc: 0.5609
Training done! Time taken: 14.408 seconds
Transforming documents...
Document transformation done! Time taken: 1.405 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7206 | train acc: 0.3615 | val loss: 1.3440 | val acc: 0.5469
Epoch 02 | train loss: 0.7841 | train acc: 0.7520 | val loss: 1.1816 | val acc: 0.5868
Epoch 03 | train loss: 0.2786 | train acc: 0.9283 | val loss: 1.3259 | val acc: 0.5815
Epoch 04 | train loss: 0.1078 | train acc: 0.9790 | val loss: 1.5143 | val acc: 0.5709
Epoch 05 | train loss: 0.0506 | train acc: 0.9922 | val loss: 1.6788 | val acc: 0.5709
Epoch 06 | train loss: 0.0317 | train acc: 0.9948 | val loss: 1.8220 | val acc: 0.5675
Epoch 07 | train loss: 0.0209 | train acc: 0.9965 | val loss: 1.9286 | val acc: 0.5602
Epoch 08 | train loss: 0.0143 | train acc: 0.9978 | val loss: 2.0230 | val acc: 0.5629
Epoch 09 | train loss: 0.0113 | train acc: 0.9977 | val loss: 2.1016 | val acc: 0.5595
Epoch 10 | train loss: 0.0121 | train acc: 0.9982 | val loss: 2.1598 | val acc: 0.5542
Training done! Time taken: 16.801 seconds
Transforming documents...
Document transformation done! Time taken: 1.497 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 1.7261 | train acc: 0.3833 | val loss: 1.3233 | val acc: 0.5629
Epoch 02 | train loss: 0.8084 | train acc: 0.7472 | val loss: 1.1692 | val acc: 0.5822
Epoch 03 | train loss: 0.2905 | train acc: 0.9222 | val loss: 1.2753 | val acc: 0.5888
Epoch 04 | train loss: 0.1155 | train acc: 0.9764 | val loss: 1.4125 | val acc: 0.5875
Epoch 05 | train loss: 0.0554 | train acc: 0.9909 | val loss: 1.5436 | val acc: 0.5835
Epoch 06 | train loss: 0.0343 | train acc: 0.9940 | val loss: 1.6824 | val acc: 0.5775
Epoch 07 | train loss: 0.0217 | train acc: 0.9972 | val loss: 1.7494 | val acc: 0.5729
Epoch 08 | train loss: 0.0183 | train acc: 0.9963 | val loss: 1.8531 | val acc: 0.5795
Epoch 09 | train loss: 0.0150 | train acc: 0.9972 | val loss: 1.9122 | val acc: 0.5782
Epoch 10 | train loss: 0.0133 | train acc: 0.9973 | val loss: 1.9752 | val acc: 0.5795
Training done! Time taken: 16.902 seconds
[array([0, 2, 1, ..., 3, 5, 0], shape=(1504,)), array([6, 2, 2, ..., 5, 0, 4], shape=(15